##### 코랩을 사용할 경우

In [ ]:
try:
    # Google Drive를 Colab(/content/drive)에 마운트
    from google.colab import drive
    drive.mount('/google_drive')

    # Colab에서 작업 디렉토리로 이동
    %cd /google_drive/Othercomputers/'내 컴퓨터'/sec05
    %ls
except ImportError:
    pass

##### 이전 노트북 실행

In [ ]:
%%capture
get_ipython().run_line_magic("run", "01_data_preprocessing.ipynb")
get_ipython().run_line_magic("run", "02_model_definition.ipynb")

##### 임포트

In [142]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

##### Device 설정

In [143]:
# GPU(CUDA) 사용 가능 시 'cuda', 아니면 'cpu' 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

사용 장치: cpu


##### 방법1: 가중치_편향만 로드하는 방법

In [144]:
# state_dict.pt 파일을 이용
pt_path = os.path.join('saved_models', 'wine_classifier_state_dict.pt')
state_dict = torch.load(pt_path, map_location=device)
model = WineClassifier().to(device)
model.load_state_dict(state_dict)

# 예측해보기
model.eval()
with torch.inference_mode():
    test_outputs  = model(test_x_tensor[0:1])
    class_label = test_outputs.argmax(dim=1)
    print(f"예측된 클래스 레이블: {class_label.item()}")


예측된 클래스 레이블: 0


##### 방법2: 모델 + 가중치_편향을 모두 로드하는 방법

In [145]:
# full.pt 파일을 이용
pt_path = os.path.join('saved_models', 'wine_classifier_full.pt')
model = torch.load(pt_path, map_location=device, weights_only=False)

# 예측해보기
model.eval()
with torch.inference_mode():  # torch.no_grad():
    test_outputs  = model(test_x_tensor[0:1])
    class_label = test_outputs.argmax(dim=1)
    print(f"예측된 클래스 레이블: {class_label.item()}")

예측된 클래스 레이블: 0


##### 방법3: 체크포인트를 이용해서 로드하는 방법

In [146]:
# checkpoint.pt 파일을 이용
pt_path = os.path.join('saved_models', 'wine_classifier_checkpoint.pt')
checkpoint = torch.load(pt_path, map_location=device, weights_only=False)
state_dict = checkpoint['model_state_dict']
model = WineClassifier().to(device)
model.load_state_dict(state_dict)

# 예측해보기
model.eval()
with torch.inference_mode():  # torch.no_grad():
    test_outputs  = model(test_x_tensor[0:1])
    class_label = test_outputs.argmax(dim=1)
    print(f"예측된 클래스 레이블: {class_label.item()}")

예측된 클래스 레이블: 0


##### 재학습하기

In [147]:
# checkpoint.pt 파일로 학습 재개하기
checkpoint_path = os.path.join('saved_models', 'wine_classifier_checkpoint.pt')
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model = WineClassifier().to(device)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer = torch.optim.Adam(model.parameters())
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch'] + 1

# 재학습해보기
model.train()
# 손실 함수(CrossEntropyLoss) 생성
loss_fn = nn.CrossEntropyLoss()
# Adam 옵티마이저(Optimizer) 생성
optimizer = optim.Adam(model.parameters(), lr=0.001)
# 학습 에포크와 배치 크기 설정
epochs = start_epoch+5
batch_size = 16
# 에포크 반복
for epoch in range(start_epoch, epochs):
    # 학습 모드로 설정: 파라미터 업데이트 및 드롭아웃 활성화
    model.train()
    # 훈련 손실과 정확도 누적 변수 초기화
    batch_loss_sum   = 0
    train_correct = 0
    # 미니배치 학습
    for i in range(0, len(train_x_tensor), batch_size):
        batch_x_tensor = train_x_tensor[i:i+batch_size]
        batch_y_tensor = train_y_tensor[i:i+batch_size]
        # 1. 기울기(그래디언트) 초기화
        optimizer.zero_grad()
        # 2. 모델 출력 얻기
        batch_train_outputs = model(batch_x_tensor)
        # 3. 손실 계산
        loss = loss_fn(batch_train_outputs, batch_y_tensor)
        # 4. 역전파: 기울기(그래디언트) 계산
        loss.backward()
        # 5. 파라미터 업데이트
        optimizer.step()
        # 각 배치 손실의 합산
        batch_loss_sum += loss.item()
        # 각 배치에서 맞춘 개수 합산: sum(): True(=1) 개수 합산
        train_correct += (batch_train_outputs.argmax(dim=1) == batch_y_tensor).sum().item()
    # 훈련 평균 손실: 각 배치 손실의 합산 / 배치 개수
    epoch_loss = batch_loss_sum / (len(train_x_tensor) / batch_size)
    # 훈련 정확도: 맞춘 개수 / 전체 개수 * 100
    epoch_acc = train_correct / len(train_x_tensor) * 100
    # 출력하기 -----------------------------------------------------
    print(f"Epoch [{epoch+1:2d}/{epochs}] "
          f"훈련 손실: {epoch_loss:.4f}  훈련 정확도: {epoch_acc:.2f}%")


Epoch [73/77] 훈련 손실: 0.0495  훈련 정확도: 99.30%
Epoch [74/77] 훈련 손실: 0.0322  훈련 정확도: 99.30%
Epoch [75/77] 훈련 손실: 0.0491  훈련 정확도: 97.89%
Epoch [76/77] 훈련 손실: 0.0200  훈련 정확도: 100.00%
Epoch [77/77] 훈련 손실: 0.0385  훈련 정확도: 99.30%
